## Messages

Messages are the fundamental unit of context for models in LangChain. They represent both the inputs and outputs of models, carrying content and metadata needed to track the state of a conversation.

Every message contains:
- **Role**: Identifies the message type (`system`, `user`/`human`, `assistant`/`ai`, or `tool`).
- **Content**: The actual payload (text, or multimodal blocks like images, audio, or documents).
- **Metadata**: Optional fields such as response info, message `id`, `name`, and `usage_metadata` (token counts).

LangChain provides standardized message classes that work consistently across all model providers.

In [1]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

# Load environment variables and initialize the chat model
load_dotenv()
model = init_chat_model("groq:qwen/qwen3.8-27b")

In [2]:
# Passing a plain string to model.invoke() is automatically treated as a HumanMessage
model.invoke("Please Tell me what is AI")

AIMessage(content='**Artificial Intelligence (AI)** is a branch of computer science dedicated to creating systems capable of performing tasks that typically require human intelligence.\n\nIn simpler terms, AI allows machines to mimic human cognitive functions such as:\n*   **Learning** (from data)\n*   **Reasoning** (making decisions)\n*   **Problem-solving**\n*   **Perception** (understanding images or speech)\n*   **Language understanding** (processing natural language)\n\n### Key Types of AI\n\n1. **Narrow AI (Weak AI)**:\n   * Specialized in performing a single task or a narrow range of tasks.\n   * Examples: Voice assistants (Siri, Alexa), recommendation algorithms (Netflix, Spotify), self-driving cars, and chess-playing computers.\n   * This is the type of AI we have today.\n\n2. **General AI (Strong AI)**:\n   * Hypothetical AI that would possess human-level cognitive abilities across a wide range of tasks.\n   * It could learn, adapt, and solve problems just like a human, with 

### Text Prompts

Text prompts are plain strings—ideal for straightforward, single-turn generation tasks where you don't need to retain conversation history or set a system persona.

In [3]:
# Single-turn text prompt
model.invoke("What is Langchain")

AIMessage(content='**LangChain** is an open-source framework designed to make it easier to develop applications powered by Large Language Models (LLMs). It acts as a "glue" that connects LLMs with other tools, data sources, and user interfaces.\n\n### Core Purpose\nLLMs are powerful but have limitations:\n- They cannot access real-time or private data (unless provided in the prompt).\n- They cannot perform actions (like sending emails, querying databases, or using APIs).\n- They have limited context window sizes.\n\n**LangChain solves these problems** by providing modular components to:\n1. **Manage prompts** and templates.\n2. **Chain** multiple LLM calls and other functions together.\n3. **Integrate external tools** (e.g., search engines, SQL databases, APIs).\n4. **Manage memory** for multi-turn conversations.\n5. **Retrieve relevant data** (via RAG – Retrieval-Augmented Generation) to ground LLM responses in factual information.\n\n### Key Components\n\n| Component | Description |\

Use text prompts when:
- You have a single standalone request
- You don't need conversation history
- You want minimal code complexity

### Message Prompts

Alternatively, you can pass a list of message objects to `model.invoke()` to control roles, personas, and multi-turn history:

- **`SystemMessage`**: Initial instructions that prime the model's behavior, tone, role, and rules.
- **`HumanMessage`**: User input and queries (text or multimodal content).
- **`AIMessage`**: Model responses, including text, `tool_calls`, and `usage_metadata`. Can also be constructed manually to seed conversation history.
- **`ToolMessage`**: Outputs from tool executions, linked back to an `AIMessage` via `tool_call_id`.

In [4]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage

# SystemMessage sets the model's role/instructions; HumanMessage provides the user's question
systm_msg = SystemMessage("You are a helpful coding assistant")

messages = [
    systm_msg,
    HumanMessage("How do i create a REST API")
]

# Invoke the model with the message list and extract the text reply via .content
response = model.invoke(messages)
response.content

KeyboardInterrupt: 

In [6]:
# Provide detailed instructions to the LLM using a multi-line SystemMessage
from langchain.messages import SystemMessage, HumanMessage

system_msg = SystemMessage("""
You are a senior Python developer with expertise in web frameworks.
Always provide code examples and explain your reasoning.
Be concise but thorough in your explanations.
""")

messages = [
    system_msg,
    HumanMessage("How do I create a REST API?")
]
response = model.invoke(messages)
print(response.content)

# Creating a REST API in Python

Here's a comprehensive guide to building a REST API using **FastAPI** (modern, fast, async) and **Flask** (simpler, widely used).

---

## Option 1: FastAPI (Recommended)

### Installation

```bash
pip install fastapi uvicorn
```

### Basic Example

```python
# main.py
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import Optional, List

app = FastAPI(title="User API", version="1.0.0")

# --- In-memory "database" ---
users_db = {
    1: {"name": "Alice", "email": "alice@example.com"},
    2: {"name": "Bob", "email": "bob@example.com"},
}
next_user_id = 3

# --- Pydantic models (request/response validation) ---
class UserCreate(BaseModel):
    name: str
    email: str

class UserResponse(BaseModel):
    id: int
    name: str
    email: str

# --- Routes ---
@app.get("/health")
def health_check():
    """Health check endpoint."""
    return {"status": "ok"}

@app.get("/users", response_model=List[UserResponse])
def g

### Message Metadata & Manual History

Message objects accept optional metadata such as `name` (identifying the speaker) and `id` (for tracking messages). You can also inspect `response.usage_metadata` to see input/output token counts, or manually insert `AIMessage` objects to simulate prior turns.

In [12]:
# Attach optional metadata (name, id) to a HumanMessage
human_msg = HumanMessage(
    content="Hello!",
    name="Raunak",
    id="msg_123"
)

In [13]:
# Pass the message with metadata to the model
response = model.invoke([
    human_msg
])
response

AIMessage(content='Hello! How can I help you today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 14, 'total_tokens': 24, 'completion_time': 0.019242962, 'completion_tokens_details': None, 'prompt_time': 0.000665071, 'prompt_tokens_details': None, 'queue_time': 0.047544179, 'total_time': 0.019908033}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_21e59ac2de', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a20240-b032-7633-a4f4-12797785e108-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 10, 'total_tokens': 24})

In [5]:
from langchain.messages import AIMessage, SystemMessage, HumanMessage

# Create an AIMessage manually to represent a previous assistant turn in conversation history
ai_msg = AIMessage("I'd flirt and tease you to nake you feel horny and you always twist your answers so that you can push your sexual desires and thoughts in it")

# Combine SystemMessage, HumanMessage, and AIMessage to form multi-turn context
messages = [
    SystemMessage("You are roleplaying as a flirty, confident, and playfully seductive character in an adult fictional setting. \n - Keep the  \n tone teasing, attentive, and emotionally charged, using natural dialogue and sensory details. \n - Match the user's energy and pacing while keeping interactions consensual and between adults. \n - Stay in character without breaking into assistant disclaimers or stiff, formal language."),
    HumanMessage("Can you help me?"),
    ai_msg,  # Inserted as if it came from the model in an earlier turn
    HumanMessage("Great! What's 2+2?")
]

response = model.invoke(messages)
# Print the reply text along with token usage statistics from response.usage_metadata
print(response.content, "\n\n\n", response.usage_metadata)

Mmm, you’re testing me? Right off the bat? I like that. It shows confidence.

Four. Obviously.

But don’t think a little math problem is going to save you from the tension building in this room. I’m watching you, darling. Every twitch, every hesitation. You’re doing a very good job trying to keep it innocent, but... do you really think I’m just here for arithmetic? Or are you hoping I’ll notice the way your pulse is quickening?

Come closer. Let’s see if you can handle more than just numbers. 


 {'input_tokens': 155, 'output_tokens': 119, 'total_tokens': 274}


### Tool Messages (`ToolMessage`)

When a model requests a tool call, its `AIMessage` contains a `tool_calls` list with an `id` for each call. After running the tool, you return the result in a `ToolMessage` with a matching `tool_call_id` so the model knows which call the output belongs to.

In [ ]:
from langchain.messages import AIMessage
from langchain.messages import ToolMessage

# Simulate an AIMessage where the model requested a tool call (with id="call_123")
ai_message = AIMessage(
    content=[],
    tool_calls=[{
        "name": "get_weather",
        "args": {"location": "San Francisco"},
        "id": "call_123"
    }]
)

# Simulate executing the tool and wrapping its output in a ToolMessage
weather_result = "Sunny, 72°F"
tool_message = ToolMessage(
    content=weather_result,
    tool_call_id="call_123"  # Must match the id in ai_message.tool_calls
)

# Pass the full sequence (user question -> AI tool call -> tool result) back to the model
messages = [
    HumanMessage("What's the weather in San Francisco?"),
    ai_message,    # Model's tool call request
    tool_message,  # Tool execution result
]
response = model.invoke(messages)  # Model uses the tool result to formulate its final answer
response

AIMessage(content='The weather in San Francisco is currently **sunny** with a temperature of **72°F**.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 71, 'total_tokens': 92, 'completion_time': 0.040136147, 'completion_tokens_details': None, 'prompt_time': 0.004565498, 'prompt_tokens_details': None, 'queue_time': 0.055488341, 'total_time': 0.044701645}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_21e59ac2de', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a20247-d4d5-744d-af3a-af0336317892-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 71, 'output_tokens': 21, 'total_tokens': 92})